**Lab06_Feature_Engineering_and_Selection**

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load ABT dataset
possible_paths = [
    Path("../data/processed/olist_orders_abt.csv"),
    Path("data/processed/olist_orders_abt.csv")
]

data_path = None
for path in possible_paths:
    if path.exists():
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError("Could not find olist_orders_abt.csv!")

df = pd.read_csv(data_path)
print(f"Dataset loaded successfully from: {data_path}")
print("Initial Shape (Rows, Columns):", df.shape)
df.head(3)

Dataset loaded successfully from: ..\data\processed\olist_orders_abt.csv
Initial Shape (Rows, Columns): (99441, 29)


,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_year,order_month,order_day,order_day_of_week,...,total_payment_value,max_payment_installments,payment_types_count,dominant_payment_type,total_items,total_price,total_freight,unique_products,unique_sellers,main_product_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017,10,2,0,...,38.71,1.0,2.0,voucher,1.0,29.99,8.72,1.0,1.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018,7,24,1,...,141.46,1.0,1.0,boleto,1.0,118.70,22.76,1.0,1.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018,8,8,2,...,179.12,3.0,1.0,credit_card,1.0,159.90,19.22,1.0,1.0,auto


In [2]:
target = "is_late_delivery"

# Ensure target column exists
if target not in df.columns:
    df[target] = (df["delivery_delay_days"] > 0).astype(int)

identifier_columns = ["order_id", "customer_id", "customer_unique_id"]

leakage_columns = [
    "delivery_days",
    "delivery_delay_days",
    "delivery_speed_category",
    "missing_delivery_days",
    "review_score",
    "review_comment_count",
    "has_review_comment",
    "is_low_review",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

columns_to_drop = [c for c in identifier_columns + leakage_columns if c in df.columns]
df = df.drop(columns=columns_to_drop)

print("Safe shape after dropping IDs and Leakage:", df.shape)
print("Remaining columns:", df.columns.tolist())

Safe shape after dropping IDs and Leakage: (99441, 20)
Remaining columns: ['customer_city', 'customer_state', 'order_status', 'order_year', 'order_month', 'order_day', 'order_day_of_week', 'order_hour', 'estimated_delivery_days', 'is_late_delivery', 'total_payment_value', 'max_payment_installments', 'payment_types_count', 'dominant_payment_type', 'total_items', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'main_product_category']


In [3]:
# 1. Average Item Price
if {"total_price", "total_items"}.issubset(df.columns):
    df["average_item_price"] = df["total_price"] / df["total_items"].replace(0, np.nan)

# 2. Freight Ratio
if {"total_freight", "total_price"}.issubset(df.columns):
    df["freight_ratio"] = df["total_freight"] / df["total_price"].replace(0, np.nan)

# 3. Items per Seller
if {"total_items", "unique_sellers"}.issubset(df.columns):
    df["items_per_seller"] = df["total_items"] / df["unique_sellers"].replace(0, np.nan)

# 4. Seller Diversity
if {"unique_sellers", "total_items"}.issubset(df.columns):
    df["seller_diversity"] = df["unique_sellers"] / df["total_items"].replace(0, np.nan)

print("Numerical ratio features created successfully!")
df[["average_item_price", "freight_ratio", "items_per_seller", "seller_diversity"]].head(3)

Numerical ratio features created successfully!


,average_item_price,freight_ratio,items_per_seller,seller_diversity
0,29.99,0.290764,1.0,1.0
1,118.70,0.191744,1.0,1.0
2,159.90,0.120200,1.0,1.0


In [4]:
# Weekend Indicator (5 = Saturday, 6 = Sunday)
if "order_day_of_week" in df.columns:
    df["is_weekend"] = (df["order_day_of_week"] >= 5).astype(int)

# Business Hours Indicator (9 AM to 6 PM)
if "order_hour" in df.columns:
    df["is_business_hour"] = (df["order_hour"].between(9, 18)).astype(int)

# Year-End Rush Indicator (November & December)
if "order_month" in df.columns:
    df["is_year_end"] = (df["order_month"].isin([11, 12])).astype(int)

print("Date & Time indicator features created!")
df[["is_weekend", "is_business_hour", "is_year_end"]].head(3)

Date & Time indicator features created!


,is_weekend,is_business_hour,is_year_end
0,0,1,0
1,0,0,0
2,0,0,0


In [5]:
# 1. Cyclical Month (12-month cycle)
if "order_month" in df.columns:
    df["month_sin"] = np.sin(2 * np.pi * df["order_month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["order_month"] / 12)

# 2. Cyclical Hour (24-hour cycle)
if "order_hour" in df.columns:
    df["hour_sin"] = np.sin(2 * np.pi * df["order_hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["order_hour"] / 24)

print("Cyclical time features (sine & cosine) created!")
df[["month_sin", "month_cos", "hour_sin", "hour_cos"]].head(3)

Cyclical time features (sine & cosine) created!


,month_sin,month_cos,hour_sin,hour_cos
0,-0.866025,0.500000,0.500000,-0.866025
1,-0.500000,-0.866025,-0.866025,0.500000
2,-0.866025,-0.500000,0.866025,-0.500000


In [6]:
# 1. Log Transformations
if "total_price" in df.columns:
    df["log_total_price"] = np.log1p(df["total_price"].clip(lower=0))

if "total_freight" in df.columns:
    df["log_total_freight"] = np.log1p(df["total_freight"].clip(lower=0))

# 2. Price Bins (Categorical Groups)
if "total_price" in df.columns:
    df["price_band"] = pd.cut(
        df["total_price"],
        bins=[-np.inf, 100, 500, 1000, 5000, np.inf],
        labels=["very_low", "low", "medium", "high", "very_high"]
    )

# 3. Interaction Feature
if {"total_items", "total_price"}.issubset(df.columns):
    df["items_x_price"] = df["total_items"] * df["total_price"]

print("Transformations, bins, and interaction features created!")

Transformations, bins, and interaction features created!


In [7]:
new_features = [
    "average_item_price", "freight_ratio", "items_per_seller", "seller_diversity",
    "is_weekend", "is_business_hour", "is_year_end",
    "month_sin", "month_cos", "hour_sin", "hour_cos",
    "log_total_price", "log_total_freight", "items_x_price"
]
new_features = [c for c in new_features if c in df.columns]

print("Missing values in engineered features before filling:")
print(df[new_features].isna().sum())

# Fill any NaN created from division with the median
for col in new_features:
    df[col] = df[col].fillna(df[col].median())

print("\nAll engineered features are clean and complete!")

Missing values in engineered features before filling:
average_item_price    775
freight_ratio         775
items_per_seller      775
seller_diversity      775
is_weekend              0
is_business_hour        0
is_year_end             0
month_sin               0
month_cos               0
hour_sin                0
hour_cos                0
log_total_price       775
log_total_freight     775
items_x_price         775
dtype: int64

All engineered features are clean and complete!


In [8]:
from sklearn.feature_selection import VarianceThreshold

numeric_df = df.select_dtypes(include=np.number).copy()
numeric_df = numeric_df.drop(columns=[target], errors="ignore").fillna(numeric_df.median())

selector = VarianceThreshold(threshold=0.0)
selector.fit(numeric_df)

selected_cols = numeric_df.columns[selector.get_support()]
print(f"Original numeric features: {len(numeric_df.columns)}")
print(f"Features after variance filtering: {len(selected_cols)}")

Original numeric features: 28
Features after variance filtering: 28


In [9]:
corr = numeric_df.corr(numeric_only=True)
threshold = 0.90

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_pairs = []

for col in upper.columns:
    for row in upper.index:
        val = upper.loc[row, col]
        if pd.notna(val) and abs(val) > threshold:
            high_corr_pairs.append((row, col, round(val, 3)))

print(f"Highly Correlated Pairs (> {threshold}):")
for r, c, v in high_corr_pairs:
    print(f" - {r} <---> {c} (Correlation: {v})")

Highly Correlated Pairs (> 0.9):
 - total_payment_value <---> total_price (Correlation: 0.985)
 - total_payment_value <---> average_item_price (Correlation: 0.911)
 - total_price <---> average_item_price (Correlation: 0.933)
 - total_items <---> items_per_seller (Correlation: 0.957)


In [10]:
from sklearn.feature_selection import mutual_info_classif

# Sample 15,000 rows so it finishes in 5 seconds
sample_df = df.sample(min(15000, len(df)), random_state=42)
mi_X = sample_df.select_dtypes(include=np.number).drop(columns=[target], errors="ignore").fillna(0)
mi_y = sample_df[target]

mi_scores = mutual_info_classif(mi_X, mi_y, random_state=42)

mi_results = pd.DataFrame({
    "feature": mi_X.columns,
    "mutual_information": mi_scores
}).sort_values("mutual_information", ascending=False)

print("Top 10 Features by Mutual Information:")
mi_results.head(10)

Top 10 Features by Mutual Information:


,feature,mutual_information
1,order_month,0.009284
22,month_cos,0.006538
21,month_sin,0.006396
15,freight_ratio,0.005919
9,total_items,0.004939
6,total_payment_value,0.004558
27,items_x_price,0.004530
25,log_total_price,0.004471
12,unique_products,0.004308
16,items_per_seller,0.004258


In [11]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(mi_X, mi_y)

importance = pd.DataFrame({
    "feature": mi_X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print("Top 10 Most Important Features (Random Forest):")
importance.head(10)

Top 10 Most Important Features (Random Forest):


,feature,importance
5,estimated_delivery_days,0.092910
11,total_freight,0.077618
26,log_total_freight,0.076293
21,month_sin,0.067999
2,order_day,0.064352
22,month_cos,0.058440
15,freight_ratio,0.056046
14,average_item_price,0.055652
6,total_payment_value,0.049177
25,log_total_price,0.048445


In [12]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "olist_orders_feature_engineered.csv"

# If running from project root, adjust path
try:
    df.to_csv(output_path, index=False)
except Exception:
    output_path = Path("data/processed/olist_orders_feature_engineered.csv")
    df.to_csv(output_path, index=False)

print(f"Engineered dataset saved successfully to: {output_path}")
print("Final Shape (Rows, Columns):", df.shape)

Engineered dataset saved successfully to: ..\data\processed\olist_orders_feature_engineered.csv
Final Shape (Rows, Columns): (99441, 35)


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

# 1. Baseline Features (Raw only)
baseline_cols = ["order_month", "order_hour", "total_price", "total_freight", "total_items"]
baseline_cols = [c for c in baseline_cols if c in df.columns]

# 2. Engineered Features (Raw + New Signals)
engineered_cols = baseline_cols + ["freight_ratio", "average_item_price", "is_weekend", "month_sin", "month_cos"]
engineered_cols = [c for c in engineered_cols if c in df.columns]

# Train-Test Split
X_base = df[baseline_cols].fillna(0)
X_eng = df[engineered_cols].fillna(0)
y_all = df[target]

X_tr_b, X_te_b, y_tr, y_te = train_test_split(X_base, y_all, test_size=0.2, random_state=42, stratify=y_all)
X_tr_e, X_te_e, _, _ = train_test_split(X_eng, y_all, test_size=0.2, random_state=42, stratify=y_all)

# Train Baseline
scaler = StandardScaler()
clf_b = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
clf_b.fit(scaler.fit_transform(X_tr_b), y_tr)
pred_b = clf_b.predict(scaler.transform(X_te_b))

# Train Engineered
clf_e = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
clf_e.fit(scaler.fit_transform(X_tr_e), y_tr)
pred_e = clf_e.predict(scaler.transform(X_te_e))

print(f"Baseline Model Accuracy   : {accuracy_score(y_te, pred_b):.4f} | F1: {f1_score(y_te, pred_b):.4f}")
print(f"Engineered Model Accuracy : {accuracy_score(y_te, pred_e):.4f} | F1: {f1_score(y_te, pred_e):.4f}")

Baseline Model Accuracy   : 0.5706 | F1: 0.1506
Engineered Model Accuracy : 0.5600 | F1: 0.1743
